# Otay Mesa Industrial-Logistics Coupling - Canyu Li

**Research question.** Is industrial activity in the Otay Mesa border area more strongly
associated with ports of entry (POEs) and major freight corridors than industrial activity
in comparable non-border industrial areas of San Diego?

**Design of this notebook.**

1. **Otay Mesa is the primary case.** Kearny Mesa, Miramar and Sorrento Valley are non-border
   comparison areas, measured with the same indicators.
2. **The unit of analysis is the census tract.** Industrial *zoning* is a coarse unit: San Diego
   is spatially constrained ("city of villages" separated by canyons), industrial zoning often
   sits on land left over after other allocations, and the city contains only a handful of
   industrial zones - too few for statistics. Census tracts give hundreds of observations and
   support real statistical comparison.
3. **Activity is measured, not just designated.** Each tract carries ACS industry-of-employment
   counts (manufacturing + wholesale + transportation/warehousing), a free and reproducible
   measure of where industrial work is located.
4. **Results are read as spatial association.** The notebook reports proximity, overlap and
   relative concentration, runs correlation/OLS, and avoids causal language.

**Data and reproducibility.** The notebook reads public sources (US Census tracts and the ACS
API, SANDAG FreightViewer, and a GADM county boundary) and needs no ArcGIS license to run its
core analysis. Census tract geometry loads from the Census TIGER service, with a GitHub-hosted
mirror as an automatic fallback when the Census host is unreachable. Optional companion datasets
(Esri GeoEnrichment by tract; geocoded business points) attach through integration hooks in
Section 10.

**Credentials are entered once**, in Section 1. A Census API key is required for the ACS request;
an ArcGIS username is optional and only used if a step needs it. Both are stored in the process
environment so no later cell asks again.

## 1. Setup - packages, folders, CRS, configuration, and credentials

`EPSG:2230` (NAD83 / California zone 6, US survey feet) is the projected CRS for all distance and
area work, so 1 mile = 5280 feet exactly. The integration hooks are enabled here; each hook checks
for its input file and degrades gracefully if the file is absent, so the notebook still runs
end-to-end on public data alone.

This cell also collects credentials a single time. Re-running it will reuse values already present
in the environment and will not prompt again.

In [ ]:
%matplotlib inline
import os
import getpass
import warnings
from pathlib import Path

import requests
import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
from shapely.geometry import Point

warnings.filterwarnings("ignore", category=UserWarning)
pd.set_option("display.max_columns", 120)

# ----------------------------------------------------------------------------
# Project folders
# ----------------------------------------------------------------------------
PROJECT_DIR    = Path.cwd()
DATA_PROCESSED = PROJECT_DIR / "data_processed"
DATA_RAW       = PROJECT_DIR / "data_raw"
FIGURES        = PROJECT_DIR / "figures"
TABLES         = PROJECT_DIR / "tables"
for d in (DATA_PROCESSED, DATA_RAW, FIGURES, TABLES):
    d.mkdir(parents=True, exist_ok=True)

# ----------------------------------------------------------------------------
# Analysis constants
# ----------------------------------------------------------------------------
TARGET_CRS  = "EPSG:2230"   # NAD83 / California zone 6 (US survey feet)
FT_PER_MILE = 5280.0
ACS_YEAR    = 2023

# Local path to the uploaded GADM county file (level-2 administrative boundaries).
GADM_FILE   = PROJECT_DIR / "gadm41_USA_2.json"

# ----------------------------------------------------------------------------
# Integration hooks: ENABLED. Each hook still checks for its input file and
# falls back cleanly if the file is missing, so the notebook is runnable alone.
# ----------------------------------------------------------------------------
USE_ESRI_ENRICHMENT = True    # data_processed/esri_enrichment_by_tract.csv  (Esri GeoEnrichment)
USE_BUSINESS_POINTS = True    # data_processed/industrial_businesses_core_geocoded.csv (geocoded points)

# ----------------------------------------------------------------------------
# Credentials entered ONCE for the whole notebook.
#   - CENSUS_API_KEY is required for the ACS request in Section 4.
#   - ARCGIS_USERNAME is optional; stored for any step that may need ArcGIS.
# Values already present in the environment are reused without prompting.
# ----------------------------------------------------------------------------
def _ensure_env(var_name, prompt, required, secret=True):
    val = os.getenv(var_name, "").strip()
    if val:
        print(f"{var_name}: using value already in environment.")
        return val
    try:
        entered = (getpass.getpass(prompt) if secret else input(prompt)).strip()
    except Exception:
        entered = ""
    if entered:
        os.environ[var_name] = entered
        print(f"{var_name}: set for this session.")
    elif required:
        raise RuntimeError(f"{var_name} is required but was not provided.")
    else:
        print(f"{var_name}: skipped (optional).")
    return entered

CENSUS_API_KEY  = _ensure_env("CENSUS_API_KEY",  "Enter Census API key: ", required=True)
ARCGIS_USERNAME = _ensure_env("ARCGIS_USERNAME", "ArcGIS username (optional, press Enter to skip): ",
                              required=False, secret=False)

print("\nProject dir :", PROJECT_DIR)
print("Target CRS  :", TARGET_CRS, "(US survey feet)")
print("ACS year    :", ACS_YEAR)
print("Hooks       : Esri =", USE_ESRI_ENRICHMENT, "| business points =", USE_BUSINESS_POINTS)
print("GADM file   :", GADM_FILE, "(exists:", GADM_FILE.exists(), ")")

## 2. Helper functions

Reusable helpers, including diagnostics that print after each spatial layer is created.
`read_esri_or_geojson_url` handles SANDAG FreightViewer files, which are Esri FeatureSet JSON
rather than GeoJSON. `download_json_to_local` fetches a remote file with `requests` and reads it
from disk, which avoids GDAL/CURL SSL issues behind some proxies. `load_tracts_with_fallback`
tries the Census TIGER host first and a GitHub-hosted Census mirror second. `load_gadm_county`
reads the uploaded GADM boundary. `style_map_axes` applies consistent cartographic styling
(title block, data-source note, simple scale bar and north arrow).

In [ ]:
def inspect_gdf(gdf, name="GeoDataFrame", n=3):
    # Quick diagnostics; run after loading any spatial layer.
    print(f"--- {name} ---")
    print("shape:", gdf.shape, "| crs:", gdf.crs)
    print("geometry types:", dict(gdf.geom_type.value_counts(dropna=False)))
    print("columns:", list(gdf.columns))
    display(gdf.head(n))

def to_target(gdf, assumed="EPSG:4326"):
    # Reproject to TARGET_CRS; set an assumed CRS first if missing.
    if gdf.crs is None:
        gdf = gdf.set_crs(assumed)
    return gdf.to_crs(TARGET_CRS)

def download_json_to_local(url, local_path, timeout=120):
    # Fetch with requests (works behind proxies that block GDAL's CURL SSL) and cache to disk.
    local_path = Path(local_path)
    if local_path.exists() and local_path.stat().st_size > 0:
        return local_path
    r = requests.get(url, timeout=timeout)
    r.raise_for_status()
    local_path.write_bytes(r.content)
    return local_path

def read_geojson_url(url, name, assumed="EPSG:4326"):
    gdf = gpd.read_file(url)
    gdf = to_target(gdf, assumed)
    print(f"[ok] {name}: {len(gdf):,} features")
    return gdf

def read_esri_or_geojson_url(url, name, assumed="EPSG:4326"):
    # SANDAG FreightViewer files are Esri FeatureSet JSON, not GeoJSON.
    # Try GeoPandas/GDAL first; if that yields nothing, parse the Esri JSON by hand.
    try:
        gdf = gpd.read_file(url)
        if len(gdf):
            print(f"[ok] {name}: {len(gdf):,} features (gdal)")
            return to_target(gdf, assumed)
    except Exception:
        pass
    from shapely.geometry import LineString, MultiLineString, Polygon, MultiPolygon
    js = requests.get(url, timeout=60).json()
    feats = js.get("features", [])
    sr = js.get("spatialReference", {}) or {}
    epsg = sr.get("latestWkid") or sr.get("wkid") or 4326
    geoms, rows = [], []
    for f in feats:
        g = f.get("geometry", {}) or {}
        attrs = f.get("attributes", {}) or {}
        geom = None
        if "x" in g and "y" in g:
            geom = Point(g["x"], g["y"])
        elif "paths" in g:
            lines = [LineString(p) for p in g["paths"] if len(p) > 1]
            geom = lines[0] if len(lines) == 1 else (MultiLineString(lines) if lines else None)
        elif "rings" in g:
            rings = [Polygon(r) for r in g["rings"] if len(r) > 2]
            geom = rings[0] if len(rings) == 1 else (MultiPolygon(rings) if rings else None)
        if geom is not None:
            geoms.append(geom); rows.append(attrs)
    gdf = gpd.GeoDataFrame(rows, geometry=geoms, crs=f"EPSG:{epsg}")
    print(f"[ok] {name}: {len(gdf):,} features (esri-json)")
    return to_target(gdf, assumed)

# Census tract sources: official TIGER first, GitHub mirror second.
TRACT_PRIMARY_URL = ("https://www2.census.gov/geo/tiger/GENZ2023/shp/"
                     "cb_2023_06_tract_500k.zip")
TRACT_MIRROR_URL  = ("https://raw.githubusercontent.com/loganpowell/census-geojson/"
                     "master/GeoJSON/500k/2020/06/tract.json")

def load_tracts_with_fallback(county_fips="073"):
    # 1) official Census TIGER cartographic boundary tracts (current vintage)
    try:
        ca = gpd.read_file(TRACT_PRIMARY_URL)
        src = "Census TIGER cb_2023 (primary)"
    except Exception as e:
        print("[fallback] Census TIGER host unreachable:", type(e).__name__, "-> GitHub mirror")
        local = download_json_to_local(TRACT_MIRROR_URL, DATA_RAW / "ca_tracts_mirror.json")
        ca = gpd.read_file(local)
        src = "GitHub census-geojson mirror (2020 500k)"
    sd = ca[ca["COUNTYFP"] == county_fips].copy()
    sd["GEOID"] = sd["GEOID"].astype(str)
    sd = sd[sd.geometry.notna() & ~sd.geometry.is_empty].copy()
    sd = to_target(sd)
    sd["area_sqmi"] = sd.geometry.area / (FT_PER_MILE ** 2)
    print(f"[tracts] source: {src} | San Diego County tracts: {len(sd):,}")
    return sd[["GEOID", "NAME", "geometry", "area_sqmi"]] if "NAME" in sd.columns \
           else sd[["GEOID", "geometry", "area_sqmi"]]

def load_gadm_county(name_1="California", name_2="SanDiego"):
    # County administrative boundary from the uploaded GADM level-2 file.
    # The southern edge of San Diego County is the US-Mexico international border,
    # which is why this boundary is useful context for the Otay Mesa maps.
    if not Path(GADM_FILE).exists():
        print(f"[gadm] {GADM_FILE} not found; county-outline overlays will be skipped.")
        return None
    g = gpd.read_file(GADM_FILE)
    sub = g[(g["NAME_1"] == name_1) & (g["NAME_2"] == name_2)].copy()
    if not len(sub):
        print("[gadm] county not found in GADM file; skipping.")
        return None
    sub = to_target(sub)
    print(f"[gadm] {name_2}, {name_1} boundary loaded ({len(sub)} feature).")
    return sub[["GID_2", "NAME_2", "geometry"]]

def add_scalebar(ax, length_mi=5, location=(0.06, 0.04)):
    # Simple scale bar in miles (data units are feet under EPSG:2230).
    x0, x1 = ax.get_xlim(); y0, y1 = ax.get_ylim()
    bar_ft = length_mi * FT_PER_MILE
    bx = x0 + (x1 - x0) * location[0]
    by = y0 + (y1 - y0) * location[1]
    ax.plot([bx, bx + bar_ft], [by, by], color="black", linewidth=3, solid_capstyle="butt", zorder=10)
    ax.text(bx + bar_ft / 2, by + (y1 - y0) * 0.012, f"{length_mi} mi",
            ha="center", va="bottom", fontsize=8, zorder=10)

def add_north_arrow(ax, location=(0.95, 0.92)):
    x0, x1 = ax.get_xlim(); y0, y1 = ax.get_ylim()
    ax.annotate("N", xy=(location[0], location[1]), xytext=(location[0], location[1] - 0.06),
                xycoords="axes fraction", ha="center", va="center", fontsize=11, fontweight="bold",
                arrowprops=dict(arrowstyle="-|>", color="black", lw=1.5), zorder=10)

def style_map_axes(ax, title, source="Sources: US Census tracts/ACS; SANDAG FreightViewer; GADM",
                   scalebar_mi=5, north=True):
    ax.set_title(title, fontweight="bold", fontsize=12, pad=10)
    ax.axis("off")
    if scalebar_mi:
        add_scalebar(ax, length_mi=scalebar_mi)
    if north:
        add_north_arrow(ax)
    ax.text(0.5, -0.02, source, transform=ax.transAxes, ha="center", va="top",
            fontsize=7, color="#555555")

def linear_fit(x, y):
    # Closed-form OLS y = a + b*x with Pearson r and R^2 (numpy only, no extra deps).
    x = np.asarray(x, float); y = np.asarray(y, float)
    m = np.isfinite(x) & np.isfinite(y)
    x, y = x[m], y[m]
    if len(x) < 3:
        return {"n": int(len(x)), "slope": np.nan, "intercept": np.nan, "r": np.nan, "r2": np.nan}
    b, a = np.polyfit(x, y, 1)
    r = float(np.corrcoef(x, y)[0, 1])
    return {"n": int(len(x)), "slope": float(b), "intercept": float(a), "r": r, "r2": r**2}

## 3. Census tracts and county boundary

San Diego County census tracts are the unit of analysis. They load from the Census TIGER
cartographic-boundary service, with a GitHub-hosted Census mirror as an automatic fallback. The
GADM level-2 file supplies a clean county administrative boundary; its southern edge coincides
with the US-Mexico international border, which is useful background for the Otay Mesa maps.

In [ ]:
tracts_sd = load_tracts_with_fallback(county_fips="073")
inspect_gdf(tracts_sd, "San Diego County census tracts")

gadm_sd = load_gadm_county("California", "SanDiego")
if gadm_sd is not None:
    inspect_gdf(gadm_sd, "GADM San Diego County boundary")

## 4. ACS industry-of-employment, by tract

The 2019-2023 ACS Data Profile (DP03) provides civilian employment by industry. The
industrial-logistics proxy is manufacturing + wholesale + transportation/warehousing, aligning
with NAICS 31-33 / 42 / 48-49.

The employment universe (denominator) is `DP03_0032E`, *civilian employed population 16+*. The
key was already collected in Section 1, so this cell reads it from the environment and does not
prompt again.

In [ ]:
# ACS industry-of-employment by tract.
# DP03_0032E civilian employed 16+ (UNIVERSE / denominator)
# DP03_0034E construction (NAICS 23; reported, not in the core proxy)
# DP03_0035E manufacturing (NAICS 31-33)
# DP03_0036E wholesale trade (NAICS 42)
# DP03_0038E transportation & warehousing, and utilities (NAICS 48-49 + 22)

CENSUS_API_KEY = os.environ.get("CENSUS_API_KEY", "").strip()
if not CENSUS_API_KEY:
    raise RuntimeError("CENSUS_API_KEY missing. Re-run the Section 1 setup cell to enter it.")

acs_vars = ["NAME", "DP03_0032E", "DP03_0034E", "DP03_0035E", "DP03_0036E", "DP03_0038E"]
base = f"https://api.census.gov/data/{ACS_YEAR}/acs/acs5/profile"
params = {"get": ",".join(acs_vars), "for": "tract:*",
          "in": "state:06 county:073", "key": CENSUS_API_KEY}

resp = requests.get(base, params=params, timeout=60)
print("Status code:", resp.status_code, "| Content-Type:", resp.headers.get("Content-Type"))

if resp.status_code != 200 or not resp.text.strip().startswith("["):
    raise RuntimeError("Census API request failed.\n\nResponse preview:\n" + resp.text[:800])

rows = resp.json()
if len(rows) <= 1:
    raise RuntimeError("Census API returned no tract-level data.")

acs = pd.DataFrame(rows[1:], columns=rows[0])
acs["GEOID"] = acs["state"] + acs["county"] + acs["tract"]

rename = {
    "DP03_0032E": "emp_total",
    "DP03_0034E": "emp_construction",
    "DP03_0035E": "emp_manufacturing",
    "DP03_0036E": "emp_wholesale",
    "DP03_0038E": "emp_transport_wh_util",
}
acs = acs.rename(columns=rename)
for c in rename.values():
    acs[c] = pd.to_numeric(acs[c], errors="coerce")

# Shared industrial-logistics definition: manufacturing + wholesale + transport/warehousing/utilities
acs["industrial_emp"] = (acs["emp_manufacturing"].fillna(0)
                         + acs["emp_wholesale"].fillna(0)
                         + acs["emp_transport_wh_util"].fillna(0))
acs["industrial_share"] = acs["industrial_emp"] / acs["emp_total"].replace(0, np.nan)

keep = ["GEOID", "NAME", "emp_total", "emp_construction", "emp_manufacturing",
        "emp_wholesale", "emp_transport_wh_util", "industrial_emp", "industrial_share"]
acs = acs[keep].copy()
print("ACS tracts:", acs.shape)
display(acs.head())

## 5. Join ACS to the tract geometries

In [ ]:
keep = ["GEOID","NAME","emp_total","emp_construction","emp_manufacturing",
        "emp_wholesale","emp_transport_wh_util","industrial_emp","industrial_share"]
tracts = tracts_sd.merge(acs[keep], on="GEOID", how="left")
tracts["industrial_emp_density"] = tracts["industrial_emp"] / tracts["area_sqmi"]
print("Tracts with ACS:", tracts.shape,
      "| missing industrial_emp:", int(tracts["industrial_emp"].isna().sum()))
display(tracts.drop(columns="geometry").head())

## 6. Freight infrastructure - ports of entry and major corridors

Ports of entry and major freight roads load from SANDAG FreightViewer, with fallbacks that keep
the analysis runnable if SANDAG is unreachable: POEs fall back to the two San Diego POEs
(San Ysidro, Otay Mesa); major roads fall back to Census TIGER primary/secondary roads.

In [ ]:
# ---- Ports of entry (with built-in fallback) -------------------------------
POE_FALLBACK = gpd.GeoDataFrame(
    {"name": ["San Ysidro POE", "Otay Mesa POE"]},
    geometry=[Point(-117.0299, 32.5421), Point(-116.9385, 32.5527)],
    crs="EPSG:4326",
).to_crs(TARGET_CRS)   # approximate coordinates; used only if SANDAG layer fails

try:
    poe = read_esri_or_geojson_url(
        "https://gis.sandag.org/FreightViewer/data/poe_points_update.json",
        "SANDAG POE points")
    if poe is None or len(poe) == 0:
        raise ValueError("empty POE layer")
    poe_source = "SANDAG FreightViewer"
except Exception as e:
    print("[fallback] SANDAG POE unavailable -> built-in POE coordinates:", e)
    poe = POE_FALLBACK.copy()
    poe_source = "built-in fallback (approximate)"
print("POE source:", poe_source, "| n =", len(poe))

# ---- Major roads / freight corridors (with Census fallback) ----------------
roads, roads_source = None, None
try:
    roads = read_esri_or_geojson_url(
        "https://gis.sandag.org/FreightViewer/data/MajorRoads_20171002.json",
        "SANDAG major roads")
    if roads is not None and len(roads):
        roads_source = "SANDAG FreightViewer major roads"
except Exception as e:
    print("[fallback] SANDAG roads unavailable:", e)

if roads is None or len(roads) == 0:
    try:
        roads = gpd.read_file(
            "https://www2.census.gov/geo/tiger/TIGER2023/PRISECROADS/tl_2023_06073_prisecroads.zip")
        roads = to_target(roads)
        if "MTFCC" in roads.columns:   # S1100 primary, S1200 secondary
            roads = roads[roads["MTFCC"].isin(["S1100", "S1200"])].copy()
        roads_source = "Census TIGER primary/secondary roads (fallback)"
    except Exception as e:
        print("[warn] no road layer; corridor indicators will be skipped:", e)

print("Roads source:", roads_source, "| n =", 0 if roads is None else len(roads))

## 7. Study areas - Otay Mesa (border) vs. non-border comparisons

Each area is a circular selection window around a known center. A tract joins an area when its
representative point falls inside that window. Otay Mesa is the border case; the other three are
non-border industrial comparisons.

In [ ]:
AREA_CENTERS = {
    "Otay Mesa":       (-117.0548, 32.5751),   # border
    "Kearny Mesa":     (-117.1397, 32.8264),   # non-border comparison
    "Miramar":         (-117.1050, 32.8970),   # non-border comparison
    "Sorrento Valley": (-117.2025, 32.8948),   # non-border comparison
}
AREA_RADIUS_MI = 3.0
BORDER_AREAS = ["Otay Mesa"]

area_windows = {}
for name, (lon, lat) in AREA_CENTERS.items():
    c = gpd.GeoDataFrame(geometry=[Point(lon, lat)], crs="EPSG:4326").to_crs(TARGET_CRS)
    area_windows[name] = c.geometry.iloc[0].buffer(AREA_RADIUS_MI * FT_PER_MILE)

tracts["area"] = "Other"
rep_pts = tracts.geometry.representative_point()
for name, win in area_windows.items():
    tracts.loc[rep_pts.within(win), "area"] = name

tracts["border"] = np.where(tracts["area"].isin(BORDER_AREAS), "Border (Otay Mesa)",
                    np.where(tracts["area"] == "Other", "Other", "Non-border comparison"))
print(tracts["area"].value_counts())

## 8. Proximity indicators for every tract

Three geometry-only indicators:
1. `dist_poe_mi` - straight-line distance from the tract's representative point to the nearest POE.
2. `dist_road_mi` - distance to the nearest major freight road.
3. `share_within_1mi_corridor` - fraction of the tract's area within 1 mile of a major road.

These describe tract-level proximity. Establishment-level distances are produced in the upstream
business-points step (Section 9); the hook in Section 10 uses those points only for per-tract counts
and validation, so the two do not duplicate each other.

In [ ]:
rep = tracts.geometry.representative_point()

# Distance to nearest POE
poe_u = poe.union_all()
tracts["dist_poe_mi"] = rep.apply(lambda g: g.distance(poe_u)) / FT_PER_MILE

# Road-based indicators (only if a road layer is available)
if roads is not None and len(roads):
    roads_u = roads.union_all()
    tracts["dist_road_mi"] = rep.apply(lambda g: g.distance(roads_u)) / FT_PER_MILE

    buf1 = roads.buffer(1 * FT_PER_MILE).union_all()
    buf_gdf = gpd.GeoDataFrame({"b": [1]}, geometry=[buf1], crs=TARGET_CRS)
    tracts["tract_area"] = tracts.geometry.area
    ov = gpd.overlay(tracts[["GEOID", "geometry", "tract_area"]], buf_gdf, how="intersection")
    ov["ov_area"] = ov.geometry.area
    s = ov.groupby("GEOID", as_index=False)["ov_area"].sum()
    tracts = tracts.merge(s, on="GEOID", how="left")
    tracts["ov_area"] = tracts["ov_area"].fillna(0)
    tracts["share_within_1mi_corridor"] = tracts["ov_area"] / tracts["tract_area"]
else:
    tracts["dist_road_mi"] = np.nan
    tracts["share_within_1mi_corridor"] = np.nan

prev = ["GEOID","area","industrial_emp","industrial_emp_density",
        "dist_poe_mi","dist_road_mi","share_within_1mi_corridor"]
display(tracts[prev].head(8))

## 9. Upstream data products (optional, ArcGIS account)

Two optional steps generate the files that the hooks in Section 10 read. Both reuse the census
tracts and freight roads already loaded above, so no layer is loaded twice, and both require an
ArcGIS account (the username entered in Section 1) with geocoding and GeoEnrichment access. Each
step writes one CSV into `data_processed/` and is controlled by a flag; if the account, the network,
or the source files are unavailable, the step reports that and is skipped, and the notebook
continues on public data.

* **9a - Geocoded industrial business points.** Reads the City business-tax CSVs, keeps the
  freight-relevant NAICS core (31-33; wholesale 421-424; transport 481-493; 8113; excluding
  425/485/486/487), geocodes the addresses, assigns each point to its tract, measures each point's
  distance to the nearest freight road, and writes `industrial_businesses_core_geocoded.csv` in
  WGS84. Hook B in Section 10 counts these per tract.
* **9b - Per-tract GeoEnrichment counts.** Submits the tracts to ArcGIS GeoEnrichment and writes
  per-tract business/employee counts as `esri_enrichment_by_tract.csv`, keyed on `GEOID`. Hook A in
  Section 10 joins these counts.

In [ ]:
# ---- 9a: geocoded industrial business points ------------------------------
# Output: data_processed/industrial_businesses_core_geocoded.csv (WGS84, with tract_geoid).
# Reuses tracts_sd (tract assignment) and roads (per-business freight distance); nothing re-loaded.
GEN_BUSINESS_POINTS = True
BUSINESS_TAX_FILES  = ["tr_active1.csv", "tr_active2.csv"]
CORE_KEEP = ("31", "32", "33", "421", "422", "423", "424",
             "481", "483", "484", "488", "491", "492", "493", "8113")
CORE_DROP = ("425", "485", "486", "487")

def _naics_group(code):
    s = str(code).strip()
    if s.startswith(("31", "32", "33")): return "Manufacturing"
    if s.startswith("42"):               return "Wholesale Trade"
    if s.startswith(("48", "49")):       return "Transportation and Warehousing"
    if s.startswith("8113"):             return "Industrial Machinery Repair"
    return "Other"

# One ArcGIS connection for the whole notebook: connect on first use, reuse afterward,
# so authentication happens at most once (the username was entered in Section 1).
_ARCGIS_CACHE = {}
def _get_gis():
    if "gis" in _ARCGIS_CACHE:
        return _ARCGIS_CACHE["gis"]
    if not ARCGIS_USERNAME:
        return None
    import arcgis
    from arcgis.gis import GIS
    gis = GIS(username=ARCGIS_USERNAME)
    arcgis.env.active_gis = gis
    _ARCGIS_CACHE["gis"] = gis
    print("ArcGIS connected as:", ARCGIS_USERNAME)
    return gis

def build_business_points():
    files = [f for f in BUSINESS_TAX_FILES if Path(f).exists()]
    if not files:
        print("[9a skip] business-tax CSVs not found; Hook B will use an existing export if present.")
        return
    if not ARCGIS_USERNAME:
        print("[9a skip] no ArcGIS username entered in Section 1; cannot geocode.")
        return
    try:
        from arcgis.geocoding import batch_geocode
        if _get_gis() is None:
            print("[9a skip] ArcGIS not connected.")
            return
    except Exception as e:
        print("[9a skip] ArcGIS unavailable:", type(e).__name__, e)
        return

    df = pd.concat([pd.read_csv(f, dtype=str, engine="python", on_bad_lines="skip") for f in files],
                   ignore_index=True)
    df.columns = (df.columns.str.strip().str.lower()
                  .str.replace(" ", "_").str.replace("#", "num"))
    df["naics"] = df["naics"].astype(str).str.strip().str.replace(".0", "", regex=False)
    core = df[df["naics"].str.startswith(CORE_KEEP, na=False)
              & ~df["naics"].str.startswith(CORE_DROP, na=False)].copy()
    core["industry_group"] = core["naics"].apply(_naics_group)
    for c in ("address", "city", "state", "zip"):
        if c not in core.columns:
            core[c] = ""
    core["full_address"] = (core["address"].fillna("") + ", " + core["city"].fillna("") + ", "
                            + core["state"].fillna("") + " " + core["zip"].fillna(""))
    core = core[core["full_address"].str.len() > 10].reset_index(drop=True)
    if core.empty:
        print("[9a skip] no core industrial addresses to geocode.")
        return

    geo = batch_geocode(core["full_address"].tolist(), out_sr=4326,
                        as_featureset=True).sdf.reset_index(drop=True)
    if len(geo) != len(core):
        print("[9a skip] geocoder row count mismatch; not writing, to avoid misalignment.")
        return
    core["longitude"]    = pd.to_numeric(geo.get("X"), errors="coerce")
    core["latitude"]     = pd.to_numeric(geo.get("Y"), errors="coerce")
    core["geocode_score"] = pd.to_numeric(geo.get("Score"), errors="coerce")
    mask = ((geo.get("Status") == "M").values
            & core["longitude"].notna().values
            & core["latitude"].notna().values)
    core = core[mask].copy()
    if core.empty:
        print("[9a skip] no addresses matched after geocoding.")
        return

    pts = gpd.GeoDataFrame(core, geometry=gpd.points_from_xy(core["longitude"], core["latitude"]),
                           crs="EPSG:4326").to_crs(TARGET_CRS)
    pts = (gpd.sjoin(pts, tracts_sd[["GEOID", "geometry"]], how="inner", predicate="within")
           .drop(columns="index_right").rename(columns={"GEOID": "tract_geoid"}))
    if roads is not None and len(roads):
        pts = gpd.sjoin_nearest(pts, roads[["geometry"]], how="left",
                                distance_col="dist_to_freight_road_ft")
        if "index_right" in pts.columns:
            pts = pts.drop(columns="index_right")
        pts["dist_to_freight_road_mi"] = pts["dist_to_freight_road_ft"] / FT_PER_MILE

    out = pts.to_crs("EPSG:4326")
    out["longitude"] = out.geometry.x
    out["latitude"]  = out.geometry.y
    cols = [c for c in ["dba_name", "naics", "industry_group", "tract_geoid",
                        "dist_to_freight_road_mi", "geocode_score",
                        "longitude", "latitude"] if c in out.columns]
    out_path = DATA_PROCESSED / "industrial_businesses_core_geocoded.csv"
    pd.DataFrame(out)[cols].to_csv(out_path, index=False)
    print(f"[9a ok] wrote {out_path.name}: {len(out):,} businesses, "
          f"{out['tract_geoid'].nunique()} tracts")

if GEN_BUSINESS_POINTS:
    build_business_points()

In [ ]:
# ---- 9b: per-tract GeoEnrichment counts -----------------------------------
# Output: data_processed/esri_enrichment_by_tract.csv (keyed on GEOID). Reuses tracts_sd.
GEN_ENRICHMENT = True
ENRICH_VARS = ["businesses.N06", "businesses.N07", "businesses.N21",
               "employees.N06", "employees.N07", "employees.N21"]

def build_enrichment():
    if not ARCGIS_USERNAME:
        print("[9b skip] no ArcGIS username entered in Section 1; cannot enrich.")
        return
    try:
        from arcgis.geoenrichment import enrich
        if _get_gis() is None:
            print("[9b skip] ArcGIS not connected.")
            return
    except Exception as e:
        print("[9b skip] ArcGIS GeoEnrichment unavailable:", type(e).__name__, e)
        return

    src = tracts_sd[["GEOID", "geometry"]].to_crs("EPSG:4326").copy()
    try:
        enriched = enrich(study_areas=src, analysis_variables=ENRICH_VARS)
    except Exception as e:
        print("[9b skip] enrich() call failed:", type(e).__name__, e)
        return
    if enriched is None or not len(enriched):
        print("[9b skip] enrichment returned no rows.")
        return

    out = pd.DataFrame(enriched)
    if "GEOID" not in out.columns and len(out) == len(src):
        out["GEOID"] = src["GEOID"].values
    keep = ["GEOID"] + [c for c in out.columns
                        if c != "GEOID" and pd.api.types.is_numeric_dtype(out[c])
                        and any(k in c.upper() for k in ("N06", "N07", "N21", "BUS", "EMP"))]
    if len(keep) <= 1:
        keep = ["GEOID"] + [c for c in out.columns
                            if c != "GEOID" and pd.api.types.is_numeric_dtype(out[c])]
    out_path = DATA_PROCESSED / "esri_enrichment_by_tract.csv"
    out[keep].to_csv(out_path, index=False)
    print(f"[9b ok] wrote {out_path.name}: {len(out):,} tracts, {len(keep) - 1} variables")

if GEN_ENRICHMENT:
    build_enrichment()

## 10. Integration hooks (enabled)

Both hooks are enabled in Section 1. Each one checks for its input file in `data_processed/` and,
if the file is absent, prints a notice and continues, so the notebook still completes on public
data alone.

* **Hook A - Esri GeoEnrichment by tract.** Adds business/employee counts per tract, joined on
  `GEOID`. Expects `data_processed/esri_enrichment_by_tract.csv`.
* **Hook B - geocoded business points.** Counts establishments per tract and reports their
  correlation with ACS industrial employment as a cross-check. Expects
  `data_processed/industrial_businesses_core_geocoded.csv` with WGS84 `longitude`/`latitude`. If
  that file already carries a `tract_geoid` column, the counts use it directly; otherwise the
  points are spatially joined to the tracts here. The hook only counts and validates - it does not
  recompute establishment-level distances.

In [ ]:
# Hook A - Esri GeoEnrichment by tract
esri_path = DATA_PROCESSED / "esri_enrichment_by_tract.csv"
if USE_ESRI_ENRICHMENT and esri_path.exists():
    enr = pd.read_csv(esri_path, dtype={"GEOID": str})
    enr.columns = [c.strip() for c in enr.columns]
    new_cols = [c for c in enr.columns if c != "GEOID"]
    tracts = tracts.merge(enr, on="GEOID", how="left")
    print("[hook A] merged Esri enrichment columns:", new_cols)
elif USE_ESRI_ENRICHMENT:
    print(f"[hook A] enabled, but {esri_path} not found -> continuing with ACS only.")
else:
    print("[hook A] disabled.")

In [ ]:
# Hook B - geocoded business points, counted per tract
biz_path = DATA_PROCESSED / "industrial_businesses_core_geocoded.csv"
if USE_BUSINESS_POINTS and biz_path.exists():
    bdf = pd.read_csv(biz_path)
    if "tract_geoid" in bdf.columns:
        # Christian's output already assigned each point to a tract.
        bdf["tract_geoid"] = bdf["tract_geoid"].astype(str)
        cnt = (bdf.groupby("tract_geoid").size()
                  .rename("business_points").reset_index()
                  .rename(columns={"tract_geoid": "GEOID"}))
        method = "tract_geoid column"
    else:
        bpts = gpd.GeoDataFrame(
            bdf, geometry=gpd.points_from_xy(bdf["longitude"], bdf["latitude"]),
            crs="EPSG:4326").to_crs(TARGET_CRS)
        joined = gpd.sjoin(bpts, tracts[["GEOID", "geometry"]], how="inner", predicate="within")
        cnt = joined.groupby("GEOID").size().rename("business_points").reset_index()
        method = "spatial join"
    cnt["GEOID"] = cnt["GEOID"].astype(str)
    tracts = tracts.merge(cnt, on="GEOID", how="left")
    tracts["business_points"] = tracts["business_points"].fillna(0)
    fit = linear_fit(tracts["business_points"], tracts["industrial_emp"])
    print(f"[hook B] counted via {method}; points vs ACS industrial employment: "
          f"r = {fit['r']:.2f} (n={fit['n']})")
elif USE_BUSINESS_POINTS:
    print(f"[hook B] enabled, but {biz_path} not found -> continuing without point counts.")
else:
    print("[hook B] disabled.")

## 11. Main analysis table

One row per tract with all indicators, saved to `tables/tract_indicators.csv`. Hook-derived
columns are included automatically when present.

In [ ]:
geom_col = tracts.geometry.name

# Standardize identifier columns if the ACS join renamed them.
if "NAME" not in tracts.columns:
    if "NAME_y" in tracts.columns:
        tracts = tracts.rename(columns={"NAME_y": "NAME"})
    elif "NAME_x" in tracts.columns:
        tracts = tracts.rename(columns={"NAME_x": "NAME"})

indicator_cols = ["GEOID"]
if "NAME" in tracts.columns:
    indicator_cols.append("NAME")
else:
    print("Note: no NAME column found; final table will identify tracts by GEOID only.")

main_indicator_cols = [
    "area", "border", "emp_total", "industrial_emp", "industrial_share",
    "industrial_emp_density", "dist_poe_mi", "dist_road_mi", "share_within_1mi_corridor",
]
# Optional columns appear only if the corresponding hook ran.
optional_indicator_cols = [
    "business_points",
    "esri_industrial_bus", "esri_industrial_emp",
]

missing_required = [c for c in main_indicator_cols if c not in tracts.columns]
if missing_required:
    print("Available columns in tracts:", list(tracts.columns))
    raise KeyError("Missing required indicator columns: " + ", ".join(missing_required))

indicator_cols += main_indicator_cols
for opt in optional_indicator_cols:
    if opt in tracts.columns:
        indicator_cols.append(opt)

analysis = tracts[indicator_cols + [geom_col]].copy()
analysis.drop(columns=geom_col).to_csv(TABLES / "tract_indicators.csv", index=False)
print("Saved:", TABLES / "tract_indicators.csv", "| rows:", len(analysis))
display(analysis.drop(columns=geom_col).head())

## 12. Statistics - association and area comparison

Exploratory, not causal. If industrial activity is drawn toward freight access, industrial
employment density should fall as distance to POE / corridors rises (negative slope and
correlation), and Otay Mesa's industrial tracts should sit closer to the POEs than the non-border
comparison areas.

In [ ]:
# (1) County-wide association: density vs proximity
print("== County-wide association (all SD tracts) ==")
for xcol in ["dist_poe_mi", "dist_road_mi"]:
    fit = linear_fit(tracts[xcol], tracts["industrial_emp_density"])
    print(f"  density ~ {xcol:14s}: slope={fit['slope']:10.1f}/mi  "
          f"r={fit['r']:.3f}  R^2={fit['r2']:.3f}  n={fit['n']}")

corr_cols = ["industrial_emp_density","industrial_share",
             "dist_poe_mi","dist_road_mi","share_within_1mi_corridor"]
corr = tracts[corr_cols].corr()
print("\nCorrelation matrix:")
display(corr.round(3))
corr.to_csv(TABLES / "correlation_matrix.csv")

In [ ]:
# (2) Border vs non-border comparison areas (per-tract means)
print("== Mean per tract, by study area ==")
sub = tracts[tracts["area"] != "Other"]
grp = (sub.groupby("area")[["industrial_emp_density","dist_poe_mi",
                            "dist_road_mi","share_within_1mi_corridor"]]
          .mean().round(2))
display(grp)
grp.to_csv(TABLES / "area_comparison_means.csv")

print("\n== Border vs non-border (per-tract means) ==")
grp2 = (sub.groupby("border")[["industrial_emp_density","dist_poe_mi",
                               "dist_road_mi","share_within_1mi_corridor"]]
           .mean().round(2))
display(grp2)

## 13. Exploratory scatter - density vs. proximity

In [ ]:
colors = {"Otay Mesa":"#d7191c","Kearny Mesa":"#2c7bb6",
          "Miramar":"#fdae61","Sorrento Valley":"#1a9641","Other":"#d9d9d9"}
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
for ax, xcol, xlab in [(axes[0], "dist_poe_mi", "Distance to nearest POE (mi)"),
                       (axes[1], "dist_road_mi", "Distance to major road (mi)")]:
    for a, col in colors.items():
        s = tracts[tracts["area"] == a]
        ax.scatter(s[xcol], s["industrial_emp_density"], s=16, alpha=0.6, c=col,
                   label=a, zorder=(1 if a == "Other" else 3), edgecolor="none")
    ax.set_xlabel(xlab); ax.set_ylabel("Industrial employment density (per sq mi)")
    ax.grid(alpha=0.2)
axes[0].legend(fontsize=8, framealpha=0.9)
fig.suptitle("Industrial employment density vs. freight proximity - San Diego tracts",
             fontweight="bold")
plt.tight_layout()
plt.savefig(FIGURES / "scatter_density_vs_proximity.png", dpi=200, bbox_inches="tight")
plt.show()

## 14. Map set (census tracts with GADM county boundary)

Three maps. The GADM county boundary is drawn on each, because the county's southern edge is the
US-Mexico border and provides context for the Otay Mesa case. The local Otay Mesa maps show the
portion of that boundary inside the window; the county-wide map clips the tract choropleth to the
GADM polygon for a clean outline and overlays the study-area windows. Each map carries a title, a
scale bar, a north arrow, and a data-source note.

In [ ]:
# Map 1 - Otay Mesa context: tracts, roads, POE, with GADM boundary (incl. intl. border)
otay_ctx = gpd.GeoDataFrame(
    geometry=[area_windows["Otay Mesa"].buffer(2 * FT_PER_MILE)], crs=TARGET_CRS)
tr_otay = gpd.clip(tracts, otay_ctx)

fig, ax = plt.subplots(figsize=(10, 9))
tr_otay.plot(ax=ax, color="#f3f0ea", edgecolor="#c9c2b8", linewidth=0.4, zorder=1)
if gadm_sd is not None:
    gpd.clip(gadm_sd.boundary, otay_ctx).plot(ax=ax, color="#111111", linewidth=1.8,
                                              zorder=4, label="County / international boundary")
if roads is not None and len(roads):
    gpd.clip(roads, otay_ctx).plot(ax=ax, color="#7a7a7a", linewidth=1.2, zorder=3,
                                   label="Major roads")
gpd.clip(poe, otay_ctx).plot(ax=ax, color="#d7191c", marker="*", markersize=130,
                             zorder=5, label="Port of entry")
ax.legend(loc="upper right", fontsize=8, framealpha=0.9)
style_map_axes(ax, "Map 1 - Otay Mesa: census tracts and freight infrastructure", scalebar_mi=2)
plt.tight_layout(); plt.savefig(FIGURES / "map1_otay_infrastructure.png", dpi=200,
                                bbox_inches="tight")
plt.show()

In [ ]:
# Map 2 - Industrial employment density (Otay Mesa) with GADM boundary
fig, ax = plt.subplots(figsize=(10, 9))
tr_otay.plot(ax=ax, column="industrial_emp_density", cmap="YlOrRd", legend=True,
             edgecolor="white", linewidth=0.3,
             legend_kwds={"label": "Industrial employment density (per sq mi)", "shrink": 0.6},
             missing_kwds={"color": "lightgray", "label": "No data"}, zorder=1)
if gadm_sd is not None:
    gpd.clip(gadm_sd.boundary, otay_ctx).plot(ax=ax, color="#111111", linewidth=1.8, zorder=4)
if roads is not None and len(roads):
    gpd.clip(roads, otay_ctx).plot(ax=ax, color="black", linewidth=0.8, zorder=3)
gpd.clip(poe, otay_ctx).plot(ax=ax, color="blue", marker="*", markersize=110, zorder=5)
style_map_axes(ax, "Map 2 - Industrial employment density by tract (Otay Mesa)", scalebar_mi=2)
plt.tight_layout(); plt.savefig(FIGURES / "map2_industrial_density.png", dpi=200,
                                bbox_inches="tight")
plt.show()

In [ ]:
# Map 3 - Distance to nearest POE (full county): tract choropleth clipped to GADM boundary
if gadm_sd is not None:
    county_poly = gadm_sd.geometry.union_all()
    tracts_clip = gpd.clip(tracts, county_poly)
else:
    tracts_clip = tracts

fig, ax = plt.subplots(figsize=(10, 10))
tracts_clip.plot(ax=ax, column="dist_poe_mi", cmap="viridis_r", legend=True,
                 edgecolor="white", linewidth=0.1,
                 legend_kwds={"label": "Distance to nearest POE (mi)", "shrink": 0.6}, zorder=2)
if gadm_sd is not None:
    gadm_sd.boundary.plot(ax=ax, color="#111111", linewidth=1.5, zorder=4)
poe.plot(ax=ax, color="red", marker="*", markersize=110, zorder=6)
for name, win in area_windows.items():
    gpd.GeoSeries([win], crs=TARGET_CRS).boundary.plot(
        ax=ax, color="black", linewidth=1.0, linestyle="--", zorder=5)
style_map_axes(ax, "Map 3 - Distance to nearest port of entry (San Diego County tracts)",
               scalebar_mi=10)
plt.tight_layout(); plt.savefig(FIGURES / "map3_distance_to_poe.png", dpi=200,
                                bbox_inches="tight")
plt.show()

## 15. Overlap measures - industrial zoning, general-plan land use, employment centers, and freight infrastructure

This section measures how four planning and infrastructure layers coincide in space, region-wide
and inside each study area.

**Layers.**
- *Industrial zoning* - City of San Diego zoning (codes IP / IL / IH / IS / IBT). Source:
  https://data.sandiego.gov/datasets/zoning/
- *Industrial general-plan land use* - City of San Diego General Plan, industrial / employment
  categories. Source: https://data.sandiego.gov/datasets/general-plan-land-use/
- *Employment centers* - SANDAG Employment Centers. Source:
  https://opendata.sandag.org/stories/s/Employment-Centers-V2-Landing-Page/grty-wn99/
- *Freight infrastructure* - ports of entry and major roads loaded above, optionally extended with
  additional FreightViewer layers, expressed as a 1-mile freight-corridor buffer for areal overlap.

**Definition.** "Industrially relevant land" is industrial zoning combined with industrial land
use (union); the zoning vs land-use agreement is reported separately.

**Measures.** For each layer pair: intersection area, the Jaccard index (area-intersection /
area-union) and containment (area-intersection / area-A). A 4-way coupling core
(relevant land intersect employment centers intersect freight corridor) is computed, and all
measures are aggregated to the census tract so they join the per-tract table.

Each loader has a fallback; a portal outage reduces the section gracefully rather than stopping
the notebook. High overlap indicates spatial coincidence, not a causal relationship.

In [ ]:
# ---- Section 15: load the four layer families ------------------------------
CORRIDOR_BUFFER_MI = 1.0
IND_ZONE_PREFIXES = ("IP", "IL", "IH", "IS", "IBT")
IND_LU_KEYWORDS = ("industrial", "employment", "business park", "office", "manufactur", "warehouse")

ZONING_URLS  = ["https://geo.sandag.org/server/rest/directories/downloads/Zoning_Base_SD.geojson"]
LANDUSE_URLS = ["https://geo.sandag.org/server/rest/directories/downloads/GeneralPlan_LandUse_SD.geojson"]
EMP_URLS     = ["https://opendata.sandag.org/api/geospatial/grty-wn99?method=export&format=GeoJSON"]
EXTRA_FREIGHT_URLS = {
    "rail_yards": "https://gis.sandag.org/FreightViewer/data/rail_yards_poly_dom.json",
    "marine":     "https://gis.sandag.org/FreightViewer/data/MarineTerminals.json",
}

def _load_first(urls, name):
    for u in urls:
        try:
            g = read_esri_or_geojson_url(u, name)
            if g is not None and len(g):
                return g
        except Exception as e:
            print(f"[..] {name}: {type(e).__name__} <- {u}")
    print(f"[warn] {name}: unavailable -> skipped")
    return None

def _pick_col(g, keys):
    return next((c for c in g.columns if any(k in c.lower() for k in keys)), None)

# industrial zoning
ind_zone = None
_z = _load_first(ZONING_URLS, "City zoning")
if _z is not None:
    _zc = _pick_col(_z, ["zone"])
    if _zc:
        ind_zone = _z[_z[_zc].astype(str).str.upper().str.strip()
                      .str.startswith(IND_ZONE_PREFIXES)][["geometry"]].copy()
        print("industrial zoning polygons:", len(ind_zone), f"(col '{_zc}')")

# industrial / employment general-plan land use
ind_lu = None
_lu = _load_first(LANDUSE_URLS, "General Plan land use")
if _lu is not None:
    _lc = _pick_col(_lu, ["landuse", "land_use", "lu_", "desc", "category", "type", "gplu"])
    if _lc:
        _m = _lu[_lc].astype(str).str.lower().str.contains("|".join(IND_LU_KEYWORDS), na=False)
        ind_lu = _lu[_m][["geometry"]].copy()
        print("industrial land-use polygons:", len(ind_lu),
              "| categories:", sorted(_lu.loc[_m, _lc].astype(str).unique())[:10])

# employment centers
emp_centers = None
_ec = _load_first(EMP_URLS, "SANDAG employment centers")
if _ec is not None:
    emp_centers = _ec[["geometry"]].copy()

# freight infrastructure -> 1-mile corridor buffer (reuse POE + roads, add extras if available)
from shapely.ops import unary_union
freight_geoms = []
for _g in (poe, roads):
    if _g is not None and len(_g):
        freight_geoms.append(_g.buffer(CORRIDOR_BUFFER_MI * FT_PER_MILE).union_all())
for _k, _u in EXTRA_FREIGHT_URLS.items():
    try:
        _g = read_esri_or_geojson_url(_u, f"freight/{_k}")
        if _g is not None and len(_g):
            freight_geoms.append(_g.buffer(CORRIDOR_BUFFER_MI * FT_PER_MILE).union_all())
    except Exception:
        pass
freight_buffer = (gpd.GeoDataFrame({"name": ["freight_corridor"]},
                                   geometry=[unary_union(freight_geoms)], crs=TARGET_CRS)
                  if freight_geoms else None)

print("layers ready:",
      {"zoning": ind_zone is not None, "landuse": ind_lu is not None,
       "emp_centers": emp_centers is not None, "freight_buffer": freight_buffer is not None})

In [ ]:
# ---- Section 15: relevant land + pairwise overlap measures -----------------
def _ua(g):
    return g.union_all()

def industrially_relevant_land(zoning, landuse):
    parts = [g for g in (zoning, landuse) if g is not None and len(g)]
    if not parts:
        return None
    geom = gpd.GeoDataFrame(pd.concat(parts, ignore_index=True), crs=TARGET_CRS).union_all()
    return gpd.GeoDataFrame({"name": ["relevant_land"]}, geometry=[geom], crs=TARGET_CRS)

def pairwise_overlap(a, b, na, nb, clip_to=None):
    if a is None or b is None or not len(a) or not len(b):
        return None
    ag, bg = _ua(a), _ua(b)
    if clip_to is not None:
        ag, bg = ag.intersection(clip_to), bg.intersection(clip_to)
    inter, union, aa, ab = ag.intersection(bg).area, ag.union(bg).area, ag.area, bg.area
    return {"layer_A": na, "layer_B": nb,
            "area_A_sqmi": aa / FT_PER_MILE**2, "area_B_sqmi": ab / FT_PER_MILE**2,
            "intersection_sqmi": inter / FT_PER_MILE**2,
            "jaccard": (inter / union) if union else np.nan,
            "containment_A_in_B": (inter / aa) if aa else np.nan,
            "containment_B_in_A": (inter / ab) if ab else np.nan}

relevant_land = industrially_relevant_land(ind_zone, ind_lu)

zlu = pairwise_overlap(ind_zone, ind_lu, "industrial_zoning", "industrial_landuse")
if zlu:
    print(f"Zoning vs land-use agreement: Jaccard={zlu['jaccard']:.3f} | "
          f"zoning inside land use={zlu['containment_A_in_B']:.3f}")
else:
    print("Zoning/land-use agreement not computable (a layer is missing).")

polys = {}
if relevant_land  is not None: polys["relevant_land"] = relevant_land
if emp_centers    is not None: polys["employment_centers"] = emp_centers
if freight_buffer is not None: polys["freight_corridor"] = freight_buffer
if ind_zone       is not None: polys["industrial_zoning"] = ind_zone
if ind_lu         is not None: polys["industrial_landuse"] = ind_lu

_rows = []
_names = list(polys)
_scopes = {"_REGION_": None}
_scopes.update(area_windows)
for _area, _win in _scopes.items():
    for _i in range(len(_names)):
        for _j in range(_i + 1, len(_names)):
            _m = pairwise_overlap(polys[_names[_i]], polys[_names[_j]],
                                  _names[_i], _names[_j], clip_to=_win)
            if _m:
                _rows.append({"area": _area, **_m})
overlap_tbl = pd.DataFrame(_rows)
if len(overlap_tbl):
    overlap_tbl.to_csv(TABLES / "overlap_pairwise_by_area.csv", index=False)
    _show = overlap_tbl[overlap_tbl["area"].isin(["_REGION_"] + BORDER_AREAS)]
    print("Pairwise overlap (region + Otay Mesa):")
    display(_show[["area", "layer_A", "layer_B", "jaccard", "containment_A_in_B"]].round(3))
else:
    print("No polygon layers available -> overlap table empty (tract analysis is unaffected).")

In [ ]:
# ---- Section 15: 4-way coupling core + aggregate overlaps to tracts --------
def coupling_core(irl, emp, fbuf):
    parts = [g for g in (irl, emp, fbuf) if g is not None and len(g)]
    if len(parts) < 2:
        return None
    geom = _ua(parts[0])
    for g in parts[1:]:
        geom = geom.intersection(_ua(g))
    return None if geom.is_empty else gpd.GeoDataFrame(
        {"name": ["coupling_core"]}, geometry=[geom], crs=TARGET_CRS)

def share_per_tract(tr, layer):
    if layer is None or not len(layer):
        return pd.Series(np.nan, index=tr.index)
    lyr = gpd.GeoDataFrame({"_k": [1]}, geometry=[_ua(layer)], crs=TARGET_CRS)
    base = tr[["GEOID", "geometry"]].copy()
    base["ta"] = base.geometry.area
    ov = gpd.overlay(base, lyr, how="intersection")
    if not len(ov):
        return pd.Series(0.0, index=tr.index)
    ov["oa"] = ov.geometry.area
    s = ov.groupby("GEOID")["oa"].sum()
    mg = base.merge(s.rename("a"), on="GEOID", how="left")
    mg["a"] = mg["a"].fillna(0.0)
    return pd.Series((mg["a"] / mg["ta"]).values, index=tr.index)

core = coupling_core(relevant_land, emp_centers, freight_buffer)

tracts["share_ind_zoning"]       = share_per_tract(tracts, ind_zone).values
tracts["share_ind_landuse"]      = share_per_tract(tracts, ind_lu).values
tracts["share_relevant_land"]    = share_per_tract(tracts, relevant_land).values
tracts["share_emp_center"]       = share_per_tract(tracts, emp_centers).values
tracts["share_freight_corridor"] = share_per_tract(tracts, freight_buffer).values
tracts["share_coupling_core"]    = share_per_tract(tracts, core).values

ov_cols = ["share_ind_zoning", "share_ind_landuse", "share_relevant_land",
           "share_emp_center", "share_freight_corridor", "share_coupling_core"]
tracts[["GEOID", "area"] + ov_cols].to_csv(TABLES / "overlap_shares_by_tract.csv", index=False)

print("Per-tract overlap shares added. Mean share by study area:")
display(tracts[tracts["area"] != "Other"].groupby("area")[ov_cols].mean().round(3))

In [ ]:
# ---- Section 15: overlap maps ----------------------------------------------
# Map 4 - overlay of the four layer families for Otay Mesa
fig, ax = plt.subplots(figsize=(10, 9))
def _clip(g):
    return gpd.clip(g, otay_ctx) if (g is not None and len(g)) else None

gpd.clip(tracts, otay_ctx).boundary.plot(ax=ax, color="#dddddd", linewidth=0.3, zorder=1)
if gadm_sd is not None:
    gpd.clip(gadm_sd.boundary, otay_ctx).plot(ax=ax, color="#111111", linewidth=1.6, zorder=6)
_rl = _clip(relevant_land)
if _rl is not None and len(_rl):
    _rl.plot(ax=ax, color="#8c6bb1", alpha=0.55, zorder=2, label="Industrially relevant land")
_ec2 = _clip(emp_centers)
if _ec2 is not None and len(_ec2):
    _ec2.boundary.plot(ax=ax, color="#1a9850", linewidth=1.4, zorder=3, label="Employment centers")
_fb = _clip(freight_buffer)
if _fb is not None and len(_fb):
    _fb.boundary.plot(ax=ax, color="#7a7a7a", linewidth=0.9, linestyle="--", zorder=3,
                      label="Freight corridor (1 mi)")
_cc = _clip(core)
if _cc is not None and len(_cc):
    _cc.plot(ax=ax, color="#fee08b", alpha=0.9, zorder=4, label="Coupling core (4-way)")
gpd.clip(poe, otay_ctx).plot(ax=ax, color="#d73027", marker="*", markersize=120, zorder=5,
                             label="Port of entry")
ax.legend(loc="upper right", fontsize=8, framealpha=0.9)
style_map_axes(ax, "Map 4 - Overlap of industrial land, employment centers & freight (Otay Mesa)",
               scalebar_mi=2)
plt.tight_layout(); plt.savefig(FIGURES / "map4_overlap_otay.png", dpi=200, bbox_inches="tight")
plt.show()

# Otay vs non-border comparison: share of relevant land inside the freight corridor
if len(overlap_tbl):
    _pf = overlap_tbl[(overlap_tbl["layer_A"] == "relevant_land")
                      & (overlap_tbl["layer_B"] == "freight_corridor")
                      & (overlap_tbl["area"] != "_REGION_")]
    if len(_pf):
        _order = ["Otay Mesa", "Kearny Mesa", "Miramar", "Sorrento Valley"]
        _pf = _pf.set_index("area").reindex(_order).dropna(subset=["containment_A_in_B"])
        fig, ax = plt.subplots(figsize=(8, 4.5))
        ax.bar(_pf.index, _pf["containment_A_in_B"],
               color=["#d7191c" if a in BORDER_AREAS else "#2c7bb6" for a in _pf.index])
        ax.set_ylabel("Relevant land inside 1-mi freight corridor")
        ax.set_title("Industrially relevant land inside freight corridor - by area", fontweight="bold")
        ax.set_ylim(0, 1); ax.grid(axis="y", alpha=0.2)
        plt.tight_layout(); plt.savefig(FIGURES / "overlap_area_comparison.png", dpi=200,
                                        bbox_inches="tight")
        plt.show()

## 16. Interpretation and limitations

**Reading the results.** Two converging lines of evidence:

1. *Proximity and activity (Sections 11-12).* A negative slope for `density ~ dist_poe_mi` and a
   lower mean `dist_poe_mi` for Otay Mesa than for the non-border comparison areas indicate that
   industrial activity in the border zone is spatially associated with port access; `dist_road_mi`
   and the corridor area-share point the same way for freight roads.
2. *Overlap measures (Section 15).* A higher containment of industrially relevant land inside the
   freight corridor, and a larger coupling-core share per tract, in Otay Mesa than in the
   comparison areas would mean the four planning/infrastructure layers coincide more tightly there.
   The zoning vs land-use Jaccard shows how far the two administrative definitions of "industrial"
   agree on the ground.

Together these describe a border-area industrial geometry that is both closer to and more
overlapping with freight infrastructure than comparable non-border industrial areas.

**Association language, not causal claims.**
> *"Industrial activity in Otay Mesa shows a structured spatial association with, and overlap with,
> border-serving freight infrastructure, stronger than in comparable non-border industrial areas."*

**Limitations.**
- *Land-allocation confounding.* San Diego's industrial land is partly what remained after other
  allocations, so proximity and overlap may be partly incidental rather than chosen for freight access.
- *Overlap depends on layer definitions.* The zoning prefixes (IP/IL/IH/IS/IBT) and the land-use
  keyword filter are documented choices; changing them changes the measured overlap.
- *Scale and granularity differ across layers.* Zoning polygons, employment-center polygons, and the
  freight buffer represent space at different resolutions, so Jaccard/containment values are
  comparative rather than absolute.
- *ACS measures workers, not facilities.* Employment is summarized by tract and does not pinpoint
  plants; `DP03_0038E` bundles utilities with transportation/warehousing (detail table `C24030`
  separates them).
- *Straight-line distance* is a simplification; network or drive-time distance would be stronger.
- *Study windows are circular selections;* results can shift with radius and center.
- *Edge effect.* The data describe only the US side of an international boundary.
- *Tract geometry vintage.* When the Census host is reachable the current TIGER tracts are used;
  the GitHub mirror provides 2020 500k tracts, whose boundaries are nearly identical for this purpose.
- *Comparisons are descriptive;* the OLS here is exploratory, not a causal model.